# 03. Iterators and Data Stream Designs

Designs where you hold **state between calls**: iterators that produce values lazily, systems that record events, feeds that merge streams.

**Learning order:** NESTED ITERATOR → PEEKING ITERATOR → SNAPSHOT ARRAY → UNDERGROUND SYSTEM → TWITTER FEED → INTERVIEW THINKING

## 1. Flatten Nested List Iterator

`[[1,1],2,[1,1]]` should produce `1,1,2,1,1` one value at a time, **lazily** (do not flatten everything up front). Keep a **stack** of pending items (reversed so the next one is on top). `has_next` expands lists on top until an integer is on top.

In [1]:
from bisect import bisect_right
from collections import defaultdict
import heapq

class NestedIterator:
    def __init__(self, nested):
        self.stack = nested[::-1]

    def has_next(self):
        while self.stack:
            top = self.stack[-1]
            if isinstance(top, int):
                return True
            self.stack.pop()
            self.stack.extend(top[::-1])             # expand the list, keeping the order
        return False

    def next(self):
        self.has_next()
        return self.stack.pop()

def flatten(nested):
    it, out = NestedIterator(nested), []
    while it.has_next():
        out.append(it.next())
    return out

print(flatten([[1, 1], 2, [1, 1]]), flatten([1, [4, [6]]]))   # [1, 1, 2, 1, 1] [1, 4, 6]

[1, 1, 2, 1, 1] [1, 4, 6]


## 2. Peeking Iterator

Add `peek()` (look at the next value without consuming it) to a normal iterator. Keep **one value in advance**: `peek` returns it, `next` returns it and loads the following one.

In [2]:
class PeekingIterator:
    _END = object()

    def __init__(self, iterator):
        self.it = iterator
        self.buffer = next(self.it, self._END)

    def peek(self):
        return self.buffer

    def next(self):
        value = self.buffer
        self.buffer = next(self.it, self._END)
        return value

    def has_next(self):
        return self.buffer is not self._END

p = PeekingIterator(iter([1, 2, 3]))
print(p.next(), p.peek(), p.next(), p.next(), p.has_next())   # 1 2 2 3 False

1 2 2 3 False


## 3. Snapshot Array

`set(i, v)`, `snap()` returns a snapshot id, `get(i, snap_id)` returns the value of index `i` **at that snapshot**. Copying the whole array on every snapshot is too expensive. Store for each index a **history** `(snap_id, value)` and use **binary search** on `snap_id`.

In [3]:
class SnapshotArray:
    def __init__(self, length):
        self.history = [[(0, 0)] for _ in range(length)]     # per index: (snap_id, value)
        self.snap_id = 0

    def set(self, index, val):
        h = self.history[index]
        if h[-1][0] == self.snap_id:
            h[-1] = (self.snap_id, val)                      # overwrite within the same snapshot period
        else:
            h.append((self.snap_id, val))

    def snap(self):
        self.snap_id += 1
        return self.snap_id - 1

    def get(self, index, snap_id):
        h = self.history[index]
        i = bisect_right(h, (snap_id, float("inf"))) - 1
        return h[i][1]

sa = SnapshotArray(3)
sa.set(0, 5)
print(sa.snap())            # 0
sa.set(0, 6)
print(sa.get(0, 0), sa.get(0, 1) if sa.snap() == 1 else None)   # 5 6

0
5 6


## 4. Underground System (average travel time)

Track check-in per customer, and for each `(start, end)` pair the **total time and count**. Two dicts.

In [4]:
class UndergroundSystem:
    def __init__(self):
        self.checkins = {}                                   # id -> (station, time)
        self.totals = defaultdict(lambda: [0, 0])            # (start, end) -> [total time, trips]

    def check_in(self, id, station, t):
        self.checkins[id] = (station, t)

    def check_out(self, id, station, t):
        start, t0 = self.checkins.pop(id)
        entry = self.totals[(start, station)]
        entry[0] += t - t0
        entry[1] += 1

    def get_average_time(self, start, end):
        total, trips = self.totals[(start, end)]
        return total / trips

u = UndergroundSystem()
u.check_in(45, "Leyton", 3); u.check_in(32, "Paradise", 8); u.check_in(27, "Leyton", 10)
u.check_out(45, "Waterloo", 15); u.check_out(27, "Waterloo", 20); u.check_out(32, "Cambridge", 22)
print(u.get_average_time("Paradise", "Cambridge"), u.get_average_time("Leyton", "Waterloo"))   # 14.0 11.0
u.check_in(10, "Leyton", 24)
print(u.get_average_time("Leyton", "Waterloo"))   # 11.0
u.check_out(10, "Waterloo", 38)
print(u.get_average_time("Leyton", "Waterloo"))   # 12.0

14.0 11.0
11.0
12.0


## 5. Design Twitter (news feed = merge k sorted lists)

`post_tweet`, `follow`, `unfollow`, `get_news_feed` (the 10 most recent tweets from the user and the people they follow).

- Each user has a list of `(timestamp, tweet_id)` in posting order.
- A global increasing **timestamp** orders tweets across users.
- `get_news_feed`: a **max-heap** (negated time) holding the **latest tweet of each followed user**. Pop the newest, push that user's previous tweet. Stop after 10. This is the *merge k sorted lists* pattern: O(k + 10 log k).

In [5]:
import heapq

class Twitter:
    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)                      # user -> [(time, tweet_id)]
        self.following = defaultdict(set)

    def post_tweet(self, user, tweet_id):
        self.tweets[user].append((self.time, tweet_id))
        self.time += 1

    def get_news_feed(self, user):
        heap = []
        for u in self.following[user] | {user}:
            if self.tweets[u]:
                i = len(self.tweets[u]) - 1
                t, tid = self.tweets[u][i]
                heapq.heappush(heap, (-t, tid, u, i))
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heapq.heappop(heap)
            feed.append(tid)
            if i > 0:
                t, nid = self.tweets[u][i - 1]
                heapq.heappush(heap, (-t, nid, u, i - 1))
        return feed

    def follow(self, follower, followee):
        self.following[follower].add(followee)

    def unfollow(self, follower, followee):
        self.following[follower].discard(followee)

tw = Twitter()
tw.post_tweet(1, 5)
print(tw.get_news_feed(1))   # [5]
tw.follow(1, 2); tw.post_tweet(2, 6)
print(tw.get_news_feed(1))   # [6, 5]
tw.unfollow(1, 2)
print(tw.get_news_feed(1))   # [5]

[5]
[6, 5]
[5]


## 6. Interview thinking

**Say:** "The object must keep state between calls. I choose structures per method so that each is fast: a stack for lazy flattening, a history list + binary search for snapshots, a heap to merge feeds."

**Common mistakes:** flattening eagerly (violates the lazy requirement and costs O(n) memory), copying the whole array per snapshot, forgetting to include the user's own tweets in the feed, not defining behavior for edge cases (unfollow self, unknown id).

## Quick Revision

- Lazy iterator: stack of pending items, expand when needed.
- Peek = buffer one value ahead.
- Snapshot array: per index history + `bisect`.
- Averages: store total and count per key.
- News feed = k-way merge with a heap.